[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filipecalegario/intro-programacao-python/blob/main/11_Dados_Visualizacao/Exercicio_series_historicas.ipynb)

# 📊 Aula de Pandas — Junção de Séries Históricas

Nesta aula vamos aprender a:

1. **Importar** múltiplos CSVs com formatação brasileira (vírgula decimal, ponto como separador de milhar)
2. **Tratar** e padronizar cada DataFrame
3. **Juntar** as três séries em um único DataFrame usando datas como chave
4. **Visualizar** as séries em um gráfico de linhas
5. **Calcular** a correlação entre os ativos

### Dados utilizados
| Arquivo | Ativo |
|---|---|
| `ouro.csv` | Ouro (futuros, USD) |
| `petroleo.csv` | Petróleo Brent (futuros, USD) |
| `petr4.csv` | PETR4 (Petrobras PN, BRL) |

> Fonte: Investing.com

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

# Configurações visuais
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## 2. Leitura dos CSVs

Os arquivos do Investing.com vêm com formatação brasileira:
- Datas no formato `dd.mm.aaaa`
- Números com **ponto** como separador de milhar e **vírgula** como decimal (ex: `5.202,51`)
- Colunas entre aspas

Vamos criar uma **função auxiliar** para ler e tratar cada arquivo de forma padronizada.

In [4]:
def ler_investing_csv(caminho: str, nome_coluna: str) -> pd.DataFrame:
    """
    Lê um CSV do Investing.com (formato BR) e retorna um DataFrame
    com DatetimeIndex e uma única coluna de preço de fechamento.

    Parâmetros
    ----------
    caminho : str
        Caminho para o arquivo CSV.
    nome_coluna : str
        Nome que a coluna de preço receberá no DataFrame final.
    """
    df = pd.read_csv(caminho, encoding='utf-8', decimal=',')

    df['Data'] = pd.to_datetime(df['Data'], format='%d.%m.%Y')

    # Remove o ponto de milhar manualmente e converte para float
    df[nome_coluna] = (
        df['Último']
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

    df = df[['Data', nome_coluna]].set_index('Data').sort_index()
    return df

print("Função 'ler_investing_csv' criada com sucesso! ✅")

Função 'ler_investing_csv' criada com sucesso! ✅


### 2.1 Aplicando a função nos três arquivos

In [6]:
df_ouro = ler_investing_csv('ouro.csv', 'Ouro_USD')
df_brent = ler_investing_csv('petroleo.csv', 'Brent_USD')
df_petr4 = ler_investing_csv('petr4.csv', 'PETR4_BRL')

print(f"Ouro:  {df_ouro.shape[0]} registros  |  de {df_ouro.index.min().date()} até {df_ouro.index.max().date()}")
print(f"Brent: {df_brent.shape[0]} registros  |  de {df_brent.index.min().date()} até {df_brent.index.max().date()}")
print(f"PETR4: {df_petr4.shape[0]} registros  |  de {df_petr4.index.min().date()} até {df_petr4.index.max().date()}")

Ouro:  51 registros  |  de 2026-01-01 até 2026-03-10
Brent: 48 registros  |  de 2026-01-02 até 2026-03-10
PETR4: 145 registros  |  de 2025-08-11 até 2026-03-10


### 2.2 Espiando cada DataFrame

In [7]:
df_ouro.head()

,Ouro_USD
Data,
2026-01-01,"4.361,55"
2026-01-02,"4.329,60"
2026-01-05,"4.451,50"
2026-01-06,"4.496,10"
2026-01-07,"4.462,50"


In [8]:
df_brent.head()

,Brent_USD
Data,
2026-01-02,60.75
2026-01-05,61.76
2026-01-06,60.70
2026-01-07,59.96
2026-01-08,61.99


In [9]:
df_petr4.head()

,PETR4_BRL
Data,
2025-08-11,30.72
2025-08-12,30.80
2025-08-13,30.57
2025-08-14,30.18
2025-08-15,29.50


## 3. Junção das séries em um único DataFrame

Como os três DataFrames já estão indexados por data, podemos usar o método `pd.concat` ou `.join()`.

⚠️ **Ponto de atenção:** os DataFrames podem ter datas diferentes (feriados, dias úteis distintos entre bolsas). Precisamos decidir como tratar isso:

| Estratégia | Método | O que acontece |
|---|---|---|
| **inner join** | mantém apenas datas em comum | perde dados, mas sem valores nulos |
| **outer join** | mantém todas as datas | pode gerar NaN onde não há dado |

Vamos usar **inner join** para garantir que todas as linhas tenham valores válidos.

In [10]:
# Junção com inner join (apenas datas em que TODOS os ativos têm cotação)
df = df_ouro.join(df_brent, how='inner').join(df_petr4, how='inner')

print(f"DataFrame combinado: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Período: {df.index.min().date()} a {df.index.max().date()}\n")
df.head(10)

DataFrame combinado: 46 linhas × 3 colunas
Período: 2026-01-02 a 2026-03-10



,Ouro_USD,Brent_USD,PETR4_BRL
Data,,,
2026-01-02,"4.329,60",60.75,30.71
2026-01-05,"4.451,50",61.76,30.20
2026-01-06,"4.496,10",60.70,29.64
2026-01-07,"4.462,50",59.96,29.83
2026-01-08,"4.460,70",61.99,30.20
2026-01-09,"4.500,90",63.34,30.30
2026-01-12,"4.614,70",63.87,30.36
2026-01-13,"4.599,10",65.47,31.14
2026-01-14,"4.635,70",66.52,31.99


### 3.1 Comparando: o que aconteceria com outer join?

In [11]:
df_outer = df_ouro.join(df_brent, how='outer').join(df_petr4, how='outer')

print(f"Com outer join: {df_outer.shape[0]} linhas")
print(f"Valores nulos por coluna:")
print(df_outer.isnull().sum())

Com outer join: 150 linhas
Valores nulos por coluna:
Ouro_USD      99
Brent_USD    102
PETR4_BRL      5
dtype: int64


## 4. Exploração rápida do DataFrame combinado

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 46 entries, 2026-01-02 to 2026-03-10
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ouro_USD   46 non-null     object 
 1   Brent_USD  46 non-null     float64
 2   PETR4_BRL  46 non-null     float64
dtypes: float64(2), object(1)
memory usage: 1.4+ KB


In [13]:
df.describe()

,Brent_USD,PETR4_BRL
count,46.000000,46.000000
mean,69.863696,36.139783
std,8.428204,3.849768
min,59.960000,29.640000
25%,64.275000,32.200000
50%,67.900000,37.215000
75%,70.830000,38.462500
max,98.960000,43.160000


## 5. Visualização — Gráfico de linhas

Como as escalas são muito diferentes (Ouro ~5000, Brent ~70-90, PETR4 ~35-43), vamos usar **normalização Min-Max** ou **eixos múltiplos** para que as três séries sejam comparáveis visualmente.

### 5.1 Gráfico com valores normalizados (base 100)

A normalização "base 100" é uma técnica comum no mercado financeiro: cada série começa em 100 e mostra a variação percentual acumulada.

In [14]:
# Normalização base 100: cada série dividida pelo seu primeiro valor × 100
df_norm = df / df.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_norm.index, df_norm['Ouro_USD'],  label='Ouro (USD)',  linewidth=2, color='#FFD700')
ax.plot(df_norm.index, df_norm['Brent_USD'], label='Brent (USD)', linewidth=2, color='#2E86AB')
ax.plot(df_norm.index, df_norm['PETR4_BRL'], label='PETR4 (BRL)', linewidth=2, color='#28A745')

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Base 100')

ax.set_title('Evolução Comparada — Ouro × Brent × PETR4 (Base 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Data')
ax.set_ylabel('Índice (base 100 = primeiro dia)')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

TypeError: unsupported operand type(s) for /: 'str' and 'str'

### 5.2 Gráfico com eixos Y múltiplos (valores absolutos)

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Eixo 1 — Ouro
color1 = '#FFD700'
ax1.plot(df.index, df['Ouro_USD'], color=color1, linewidth=2, label='Ouro (USD)')
ax1.set_ylabel('Ouro (USD)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)

# Eixo 2 — Brent
ax2 = ax1.twinx()
color2 = '#2E86AB'
ax2.plot(df.index, df['Brent_USD'], color=color2, linewidth=2, label='Brent (USD)')
ax2.set_ylabel('Brent (USD)', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)

# Eixo 3 — PETR4
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('axes', 1.1))  # desloca o 3º eixo para fora
color3 = '#28A745'
ax3.plot(df.index, df['PETR4_BRL'], color=color3, linewidth=2, label='PETR4 (BRL)')
ax3.set_ylabel('PETR4 (BRL)', color=color3, fontsize=12)
ax3.tick_params(axis='y', labelcolor=color3)

# Título e formatação
ax1.set_title('Cotações Diárias — Ouro × Brent × PETR4', fontsize=14, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45)

# Legenda combinada
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
lines3, labels3 = ax3.get_legend_handles_labels()
ax1.legend(lines1 + lines2 + lines3, labels1 + labels2 + labels3, loc='upper left', fontsize=11)

fig.tight_layout()
plt.show()

## 6. Análise de Correlação

### 6.1 Correlação de Pearson (nos preços)

A correlação de Pearson mede a relação linear entre duas variáveis:
- **+1**: correlação positiva perfeita
- **0**: sem correlação linear
- **-1**: correlação negativa perfeita

⚠️ **Cuidado:** correlação entre *preços* pode ser enganosa (duas séries que apenas sobem ao longo do tempo terão alta correlação sem necessariamente ter relação causal). Por isso, também vamos calcular a correlação dos **retornos diários**.

In [ ]:
# Matriz de correlação dos PREÇOS
corr_precos = df.corr()

print("Matriz de Correlação (Preços):")
print(corr_precos.round(4))

In [ ]:
# Heatmap da correlação dos preços
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_precos,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    square=True,
    linewidths=1,
    ax=ax
)
ax.set_title('Correlação de Pearson — Preços', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Correlação dos retornos diários (mais robusta)

Os retornos diários eliminam a tendência de alta/baixa e capturam melhor a relação real entre os ativos.

In [ ]:
# Calcula retornos percentuais diários
retornos = df.pct_change().dropna()

print("Primeiros retornos diários:")
retornos.head()

In [ ]:
# Matriz de correlação dos RETORNOS
corr_retornos = retornos.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_retornos,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    square=True,
    linewidths=1,
    ax=ax
)
ax.set_title('Correlação de Pearson — Retornos Diários', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Teste estatístico de significância (Pearson com p-valor)

Agora vamos além da simples matriz: usamos `scipy.stats.pearsonr` para obter o **p-valor** de cada par. Um p-valor < 0.05 indica que a correlação é estatisticamente significativa.

In [ ]:
# Teste de correlação de Pearson com p-valor para cada par de ativos
pares = [
    ('Ouro_USD', 'Brent_USD'),
    ('Ouro_USD', 'PETR4_BRL'),
    ('Brent_USD', 'PETR4_BRL'),
]

print("=" * 65)
print(f"{'Par':<25} {'Pearson r':>10} {'p-valor':>12} {'Significativo?':>16}")
print("=" * 65)

for col_a, col_b in pares:
    r, p = stats.pearsonr(retornos[col_a], retornos[col_b])
    sig = '✅ Sim' if p < 0.05 else '❌ Não'
    print(f"{col_a} × {col_b:<10} {r:>10.4f} {p:>12.6f} {sig:>16}")

print("=" * 65)
print("\n(Calculado sobre os retornos diários)")

### 6.4 Scatter plots dos retornos — visualizando a correlação

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col_a, col_b) in zip(axes, pares):
    ax.scatter(retornos[col_a], retornos[col_b], alpha=0.6, edgecolors='w', s=50)

    # Linha de regressão
    m, b = np.polyfit(retornos[col_a], retornos[col_b], 1) if True else (0, 0)
    x_line = np.linspace(retornos[col_a].min(), retornos[col_a].max(), 100)
    ax.plot(x_line, m * x_line + b, color='red', linewidth=2, linestyle='--')

    r, _ = stats.pearsonr(retornos[col_a], retornos[col_b])
    ax.set_title(f'{col_a} × {col_b}\nr = {r:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col_a)
    ax.set_ylabel(col_b)

plt.suptitle('Scatter Plot dos Retornos Diários', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Resumo da aula

### O que aprendemos hoje:

| Conceito | Função/Método |
|---|---|
| Ler CSV com formatação BR | `pd.read_csv(..., thousands='.', decimal=',')` |
| Converter datas | `pd.to_datetime(col, format='%d.%m.%Y')` |
| Definir índice | `df.set_index('Data')` |
| Juntar DataFrames por índice | `df.join(outro_df, how='inner')` |
| Normalizar para base 100 | `df / df.iloc[0] * 100` |
| Retornos diários | `df.pct_change()` |
| Correlação | `df.corr()` e `scipy.stats.pearsonr()` |

### Para ir além:
- Experimentem trocar o `how='inner'` por `how='outer'` e usar `df.fillna(method='ffill')` para preencher os NaNs
- Testem a correlação de **Spearman** (`stats.spearmanr`) para capturar relações não-lineares
- Adicionem mais ativos ao DataFrame!